# 🛰️ Satellite Footprint Visualization

This notebook computes the **Line-of-Sight footprint** of a satellite on Earth's surface
from a **TLE (Two-Line Element Set)** dataset and visualizes it on an interactive 3D globe.

### Coordinate System Overview

| Frame | Description |
|-------|-------------|
| **ECI** | Fixed in space, x-axis points toward vernal equinox |
| **ECEF** | Rotates with the Earth, x-axis points toward prime meridian |
| **Geodetic** | Lat / Lon on a perfect sphere (R = 6371 km) |

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timezone, timedelta
from sgp4.api import Satrec, jday
import plotly.graph_objects as go

## Load & Sample the TLE Dataset

The dataset is loaded from a CSV file containing orbital elements for multiple satellites.
A single satellite is randomly selected using `df.sample()` with a fixed `random_state` for reproducibility.

In [2]:
df = pd.read_csv("starlink.txt")
df.head()

,OBJECT_NAME,OBJECT_ID,EPOCH,MEAN_MOTION,ECCENTRICITY,INCLINATION,RA_OF_ASC_NODE,ARG_OF_PERICENTER,MEAN_ANOMALY,EPHEMERIS_TYPE,CLASSIFICATION_TYPE,NORAD_CAT_ID,ELEMENT_SET_NO,REV_AT_EPOCH,BSTAR,MEAN_MOTION_DOT,MEAN_MOTION_DDOT
0,STARLINK-1008,2019-074B,2026-05-03T19:38:45.492576,15.463968,0.000108,53.1551,280.3842,20.3632,339.7419,0,U,44714,999,35734,0.000391,0.000193,0.000000
1,STARLINK-1012,2019-074F,2026-05-03T21:15:55.215072,15.465733,0.000057,53.1591,280.3857,17.0362,343.0665,0,U,44718,999,35734,0.000404,0.000200,0.000000
2,STARLINK-1017,2019-074L,2026-05-03T19:10:01.904160,15.288927,0.000255,53.0489,276.2399,262.7691,97.3018,0,U,44723,999,35738,0.001902,0.000542,0.000000
3,STARLINK-1019,2019-074M,2026-05-04T02:25:42.364128,16.094867,0.001107,53.0307,213.5402,346.4662,13.6072,0,U,44724,999,35827,0.000638,0.005996,0.000197
4,STARLINK-1020,2019-074N,2026-05-03T21:11:34.967904,15.498740,0.000049,53.1607,305.6062,66.1600,293.9461,0,U,44725,999,35697,0.001489,0.000849,0.000000


In [3]:
df_random = df.sample(random_state=42)
df_random

,OBJECT_NAME,OBJECT_ID,EPOCH,MEAN_MOTION,ECCENTRICITY,INCLINATION,RA_OF_ASC_NODE,ARG_OF_PERICENTER,MEAN_ANOMALY,EPHEMERIS_TYPE,CLASSIFICATION_TYPE,NORAD_CAT_ID,ELEMENT_SET_NO,REV_AT_EPOCH,BSTAR,MEAN_MOTION_DOT,MEAN_MOTION_DDOT
9,STARLINK-1046,2019-074AQ,2026-05-03T21:52:48.599904,15.507943,0.000338,53.0504,307.0716,328.8321,31.2486,0,U,44751,999,35696,0.001324,0.00078,0.0


## Satellite Configuration & Time Window

The satellite object is initialized directly from the DataFrame using `Satrec.sgp4init()`
with individual orbital elements instead of raw TLE strings.

**Key unit conversions:**
| Parameter | CSV unit | SGP4 unit |
|-----------|----------|-----------|
| `MEAN_MOTION` | rev/day | rad/min → multiply by `2π / 1440` |
| `INCLINATION`, `RA_OF_ASC_NODE`, ... | degrees | radians → `np.radians()` |
| `EPOCH` | ISO string | days since 1949-12-31 |

Set `CUSTOM_TIME = None` to start at the TLE epoch, or provide a UTC string for a specific moment.

In [4]:
# ── Helper: convert ISO epoch string to SGP4 internal day count ───────────────
def _epoch_to_days(epoch_str):
    """Convert ISO epoch (timezone-naive) to days since 1949-12-31 (SGP4 format)."""
    dt = datetime.fromisoformat(epoch_str)
    dt = dt.replace(tzinfo=timezone.utc)
    ref = datetime(1949, 12, 31, 0, 0, 0, tzinfo=timezone.utc)
    return (dt - ref).total_seconds() / 86400.0

# ── Load satellite row from DataFrame ─────────────────────────────────────────
sat_row = df_random.iloc[0]

# ── Initialize SGP4 satellite object ──────────────────────────────────────────
sat = Satrec()
sat.sgp4init(
    2,                                                        # WGS84
    'i',                                                      # improved mode
    int(sat_row['NORAD_CAT_ID']),
    _epoch_to_days(sat_row['EPOCH']),                         # epoch [days since 1949-12-31]
    float(sat_row['BSTAR']),                                  # drag term
    float(sat_row['MEAN_MOTION_DOT']),                        # dn/dt / 2
    float(sat_row['MEAN_MOTION_DDOT']),                       # d²n/dt² / 6
    float(sat_row['ECCENTRICITY']),                           # eccentricity
    np.radians(float(sat_row['ARG_OF_PERICENTER'])),          # ω [rad]
    np.radians(float(sat_row['INCLINATION'])),                # i [rad]
    np.radians(float(sat_row['MEAN_ANOMALY'])),               # M [rad]
    float(sat_row['MEAN_MOTION']) * (2 * np.pi / 1440),       # n [rad/min]
    np.radians(float(sat_row['RA_OF_ASC_NODE'])),             # Ω [rad]
)

# ── Time window configuration ─────────────────────────────────────────────────
epoch_dt = datetime.fromisoformat(sat_row['EPOCH']).replace(tzinfo=timezone.utc)

# Set a custom UTC time (format: "YYYY-MM-DD HH:MM") or None to use the epoch
CUSTOM_TIME = None

if CUSTOM_TIME:
    START_TIME = datetime.strptime(CUSTOM_TIME, "%Y-%m-%d %H:%M").replace(tzinfo=timezone.utc)
else:
    START_TIME = epoch_dt

DURATION_MINUTES = 90    # total propagation duration in minutes
STEP_MINUTES     = 10    # time step between positions in minutes
EARTH_RADIUS_KM  = 6371.0

print(f"Satellite: {sat_row['OBJECT_NAME']}  (NORAD {sat_row['NORAD_CAT_ID']})")
print(f"Epoch:     {sat_row['EPOCH']}")
print(f"Start:     {START_TIME.strftime('%Y-%m-%d %H:%M UTC')}")
print(f"Duration:  {DURATION_MINUTES} min  |  Step: {STEP_MINUTES} min")
print(f"Earth radius: {EARTH_RADIUS_KM} km (perfect sphere)")

Satellite: STARLINK-1046  (NORAD 44751)
Epoch:     2026-05-03T21:52:48.599904
Start:     2026-05-03 21:52 UTC
Duration:  90 min  |  Step: 10 min
Earth radius: 6371.0 km (perfect sphere)


## Coordinate Transformations

### ECI → ECEF
SGP4 outputs positions in the **ECI frame** (fixed in space).
To map positions onto the rotating Earth, we apply a rotation around the z-axis
by the **Greenwich Mean Sidereal Time (GMST)** angle θ:

$$\begin{pmatrix} x \\ y \\ z \end{pmatrix}_{\text{ECEF}} = \begin{pmatrix} \cos\theta & \sin\theta & 0 \\ -\sin\theta & \cos\theta & 0 \\ 0 & 0 & 1 \end{pmatrix} \cdot \begin{pmatrix} x \\ y \\ z \end{pmatrix}_{\text{ECI}}$$

### ECEF → Geodetic (perfect sphere)
$$\text{Lat} = \arcsin\left(\frac{z}{r}\right), \quad \text{Lon} = \text{atan2}(y, x), \quad \text{Alt} = r - R_E$$

In [5]:
def gmst_from_datetime(dt):
    """Compute Greenwich Mean Sidereal Time [radians] for a given UTC datetime."""
    J2000 = datetime(2000, 1, 1, 12, 0, 0, tzinfo=timezone.utc)
    d = (dt - J2000).total_seconds() / 86400.0       # days since J2000
    gmst_deg = 280.46061837 + 360.98564736629 * d     # IAU formula
    return np.radians(gmst_deg % 360)


def eci_to_ecef(x_eci, y_eci, z_eci, gmst):
    """Rotate ECI position vector to ECEF via z-axis rotation by GMST angle."""
    cos_g, sin_g = np.cos(gmst), np.sin(gmst)
    x_ecef =  cos_g * x_eci + sin_g * y_eci
    y_ecef = -sin_g * x_eci + cos_g * y_eci
    z_ecef =  z_eci
    return x_ecef, y_ecef, z_ecef


def ecef_to_geodetic(x, y, z):
    """Convert ECEF [km] to geodetic coordinates (lat°, lon°, radius km) — perfect sphere."""
    r   = np.sqrt(x**2 + y**2 + z**2)
    lat = np.degrees(np.arcsin(z / r))
    lon = np.degrees(np.arctan2(y, x))
    return lat, lon, r


def geodetic_to_ecef(lat_deg, lon_deg, r=EARTH_RADIUS_KM):
    """Convert geodetic (lat°, lon°) to ECEF [km] on a perfect sphere."""
    lat = np.radians(lat_deg)
    lon = np.radians(lon_deg)
    x = r * np.cos(lat) * np.cos(lon)
    y = r * np.cos(lat) * np.sin(lon)
    z = r * np.sin(lat)
    return x, y, z

## SGP4 Orbit Propagation

The **SGP4 algorithm** computes the satellite position at any given time from the TLE orbital elements.
Output is a position vector **r** in ECI coordinates (km).

Each time step returns: UTC datetime, Lat/Lon (degrees), Altitude (km), and ECEF coordinates (x, y, z).

In [6]:
def propagate_satellite(sat, start_dt, duration_min, step_min):
    """
    Propagate satellite orbit over a time window using SGP4.

    Parameters
    ----------
    sat          : Satrec   — initialized SGP4 satellite object
    start_dt     : datetime — start time (UTC, timezone-aware)
    duration_min : int      — total duration in minutes
    step_min     : int      — time step between positions in minutes

    Returns
    -------
    list of (datetime, lat, lon, alt_km, x_ecef, y_ecef, z_ecef)
    """
    positions = []
    n_steps = int(duration_min / step_min) + 1

    for i in range(n_steps):
        dt = start_dt + timedelta(minutes=i * step_min)

        # Convert to Julian date for SGP4
        jd, fr = jday(dt.year, dt.month, dt.day,
                      dt.hour, dt.minute, dt.second)

        # SGP4: e = error code, r = position [km], v = velocity [km/s]
        e, r, v = sat.sgp4(jd, fr)

        if e != 0:
            print(f"⚠️  SGP4 error at {dt}: code {e}")
            continue

        # ECI → ECEF → Geodetic
        gmst             = gmst_from_datetime(dt)
        x, y, z          = eci_to_ecef(r[0], r[1], r[2], gmst)
        lat, lon, radius = ecef_to_geodetic(x, y, z)
        alt_km           = radius - EARTH_RADIUS_KM

        positions.append((dt, lat, lon, alt_km, x, y, z))

    return positions


# ── Run propagation ───────────────────────────────────────────────────────────
positions = propagate_satellite(sat, START_TIME, DURATION_MINUTES, STEP_MINUTES)

print(f"{'Time (UTC)':<20} {'Lat':>8} {'Lon':>9} {'Alt (km)':>10}")
print("-" * 52)
for dt, lat, lon, alt, *_ in positions:
    print(f"{dt.strftime('%H:%M'):<20} {lat:>+8.2f}° {lon:>+8.2f}° {alt:>10.0f}")

Time (UTC)                Lat       Lon   Alt (km)
----------------------------------------------------
21:52                   -0.03°  +117.04°        420
22:02                  +30.04°  +140.32°        417
22:12                  +51.30°  -178.04°        415
22:22                  +45.65°  -120.92°        417
22:32                  +19.51°   -88.57°        422
22:42                  -11.20°   -67.07°        425
22:52                  -39.54°   -39.77°        426
23:02                  -53.00°   +11.86°        425
23:12                  -37.54°   +61.40°        424
23:22                   -8.70°   +87.58°        421


## Footprint Calculation (Line of Sight)

The footprint is the region on Earth visible from the satellite above the horizon (elevation ≥ 0°).

### Footprint Half-Angle (Earth Central Angle)

$$\rho = \arccos\left(\frac{R_E}{R_E + h}\right)$$

The boundary is a **spherical cap** around the sub-satellite point, parametrized by azimuth α ∈ [0°, 360°].

In [7]:
def compute_footprint(sat_lat, sat_lon, sat_alt_km, n_points=180):
    """
    Compute the Line-of-Sight footprint boundary as a spherical cap.

    Parameters
    ----------
    sat_lat, sat_lon : float — satellite position in degrees
    sat_alt_km       : float — altitude above Earth's surface in km
    n_points         : int   — number of points along the footprint boundary

    Returns
    -------
    fp_lats, fp_lons : list — footprint boundary coordinates in degrees
    """
    rho       = np.arccos(EARTH_RADIUS_KM / (EARTH_RADIUS_KM + sat_alt_km))  # Earth Central Angle
    sat_lat_r = np.radians(sat_lat)
    sat_lon_r = np.radians(sat_lon)

    fp_lats, fp_lons = [], []

    for az_deg in np.linspace(0, 360, n_points):
        az = np.radians(az_deg)

        fp_lat = np.arcsin(
            np.sin(sat_lat_r) * np.cos(rho) +
            np.cos(sat_lat_r) * np.sin(rho) * np.cos(az)
        )
        fp_lon = sat_lon_r + np.arctan2(
            np.sin(az) * np.sin(rho) * np.cos(sat_lat_r),
            np.cos(rho) - np.sin(sat_lat_r) * np.sin(fp_lat)
        )

        fp_lats.append(np.degrees(fp_lat))
        fp_lons.append(np.degrees(fp_lon))

    return fp_lats, fp_lons


# ── Example: footprint stats for the first position ───────────────────────────
dt0, lat0, lon0, alt0, *_ = positions[0]
rho0 = np.degrees(np.arccos(EARTH_RADIUS_KM / (EARTH_RADIUS_KM + alt0)))

print(f"Satellite:         Lat={lat0:+.2f}°  Lon={lon0:+.2f}°  Alt={alt0:.0f} km")
print(f"Footprint angle ρ: {rho0:.2f}°  (~{2*rho0:.1f}° diameter)")
print(f"Footprint radius:  {np.radians(rho0) * EARTH_RADIUS_KM:.0f} km")

Satellite:         Lat=-0.03°  Lon=+117.04°  Alt=420 km
Footprint angle ρ: 20.27°  (~40.5° diameter)
Footprint radius:  2254 km


## 3D Globe Visualization

All elements are combined into an interactive **Plotly orthographic globe**.
`projection_type="orthographic"` renders a rotatable 3D sphere with a real world map built into Plotly — no external textures needed.

| Element | Description |
|---------|-------------|
| **World map** | Plotly built-in: land, ocean, coastlines, countries |
| **Footprint boundary** | Closed circle on Earth's surface |
| **Footprint fill** | Transparent polygon (spherical cap area) |
| **Satellite marker** | Star symbol with hover tooltip |

In [8]:
def build_figure(positions):
    fig = go.Figure()

    # Invisible base trace required to initialize the geo layout
    fig.add_trace(go.Scattergeo(
        lat=[], lon=[],
        mode="markers",
        showlegend=False,
        hoverinfo="skip",
    ))

    # Color palette: blue → red over time
    n      = len(positions)
    colors = [f"hsl({int(240 - 240 * i / max(n-1, 1))}, 90%, 60%)" for i in range(n)]

    for i, (dt, lat, lon, alt_km, sx, sy, sz) in enumerate(positions):
        color = colors[i]
        label = dt.strftime("%H:%M UTC")

        # Compute footprint boundary
        fp_lats, fp_lons = compute_footprint(lat, lon, alt_km)
        fp_lats_closed   = fp_lats + [fp_lats[0]]
        fp_lons_closed   = fp_lons + [fp_lons[0]]

        # Footprint fill + boundary
        fig.add_trace(go.Scattergeo(
            lat=fp_lats_closed,
            lon=fp_lons_closed,
            mode="lines",
            fill="toself",
            fillcolor=color.replace("hsl", "hsla").replace(")", ", 0.25)"),
            line=dict(color=color, width=2),
            name=f"Footprint {label}",
            legendgroup=f"sat_{i}",
            hoverinfo="skip",
        ))

        # Satellite marker (no text label on globe)
        fig.add_trace(go.Scattergeo(
            lat=[lat], lon=[lon],
            mode="markers",
            marker=dict(size=12, color=color, symbol="star",
                        line=dict(color="white", width=1)),
            name=f"{label}  Alt: {alt_km:.0f} km",
            legendgroup=f"sat_{i}",
            hovertemplate=(
                f"<b>{dt.strftime('%Y-%m-%d %H:%M UTC')}</b><br>"
                f"Lat: {lat:.2f}°<br>"
                f"Lon: {lon:.2f}°<br>"
                f"Alt: {alt_km:.0f} km<br>"
                "<extra></extra>"
            ),
        ))

    fig.update_layout(
        title=dict(
            text=f"🛰️  {sat_row['OBJECT_NAME']} — Footprint Visualization",
            font=dict(size=20, color="white"), x=0.5,
        ),
        paper_bgcolor="#0d1117",
        geo=dict(
            showland=True,       landcolor="#2d4a2d",
            showocean=True,      oceancolor="#1a2f4a",
            showlakes=True,      lakecolor="#1a2f4a",
            showrivers=True,     rivercolor="#1a3a5c",
            showcountries=True,  countrycolor="rgba(255,255,255,0.2)",
            showcoastlines=True, coastlinecolor="rgba(255,255,255,0.4)",
            showframe=False,
            bgcolor="#0d1117",
            projection_type="orthographic",       # rotatable 3D globe
            center=dict(lat=positions[0][1], lon=positions[0][2]),
        ),
        legend=dict(
            bgcolor="rgba(30,30,50,0.85)",
            font=dict(color="white", size=11),
            bordercolor="rgba(255,255,255,0.2)",
            borderwidth=1,
        ),
        margin=dict(l=0, r=0, t=50, b=0),
        height=750,
    )
    return fig


# ── Render plot ───────────────────────────────────────────────────────────────
fig = build_figure(positions)
fig.show()